# Customer Churn Prediction Using an Artificial Neural Network

This notebook documents the development of a customer churn prediction model using an Artificial Neural Network (ANN).

I built this project to understand the complete Deep Learning workflow on a real-world style tabular dataset — starting from raw customer data, preparing the features, training an ANN, monitoring the training process, and finally saving the trained model for later predictions.

The main objective is to predict whether a bank customer is likely to leave the bank (`Exited = 1`) or remain with the bank (`Exited = 0`).

### What I will cover in this notebook

- Load and inspect the customer churn dataset
- Remove irrelevant features
- Encode categorical variables
- Apply feature scaling
- Split the data into training and testing sets
- Build an Artificial Neural Network using TensorFlow/Keras
- Train the model using Binary Crossentropy and Adam optimizer
- Use Early Stopping to control overfitting
- Track training with TensorBoard
- Save the trained model and preprocessing objects for inference

The goal here is not just to train a model and look at its accuracy, but to understand how the different pieces of a Deep Learning workflow fit together and how the trained model can later be used in an application.

## 1. Import Libraries

First, I am importing the libraries needed for data manipulation, preprocessing, model development, and saving the preprocessing objects.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
import pickle

## 2. Load the Dataset

The dataset contains customer-level demographic, financial, and account information along with the target variable `Exited`.

Before building the model, I want to first look at the raw data and understand what information is available.

In [2]:
# Load the dataset

data = pd.read_csv("/workspaces/Ann-churn-prediction/data/Churn_Modelling.csv")
data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


## 3. Data Preprocessing

The raw dataset contains a few columns that are identifiers rather than useful predictive features. I will remove those first and then convert the categorical variables into numerical representations that the neural network can work with.

In [3]:
# Preprocess the data
# Drop irrelevant features

data = data.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1)
data

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0
...,...,...,...,...,...,...,...,...,...,...,...
9995,771,France,Male,39,5,0.00,2,1,0,96270.64,0
9996,516,France,Male,35,10,57369.61,1,1,1,101699.77,0
9997,709,France,Female,36,7,0.00,1,0,1,42085.58,1
9998,772,Germany,Male,42,3,75075.31,2,1,0,92888.52,1


### Handling Categorical Features

The dataset contains categorical information such as `Gender` and `Geography`. Since the ANN expects numerical input, I will transform these columns into numerical features while keeping the original information intact.

In [4]:
# Encode Categorical Variables

label_encoder_gender = LabelEncoder()
data['Gender'] = label_encoder_gender.fit_transform(data['Gender'])
data

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,0,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,0,41,1,83807.86,1,0,1,112542.58,0
2,502,France,0,42,8,159660.80,3,1,0,113931.57,1
3,699,France,0,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,0,43,2,125510.82,1,1,1,79084.10,0
...,...,...,...,...,...,...,...,...,...,...,...
9995,771,France,1,39,5,0.00,2,1,0,96270.64,0
9996,516,France,1,35,10,57369.61,1,1,1,101699.77,0
9997,709,France,0,36,7,0.00,1,0,1,42085.58,1
9998,772,Germany,1,42,3,75075.31,2,1,0,92888.52,1


In [5]:
# OneHot Encode 'Geography'

from sklearn.preprocessing import OneHotEncoder

one_hot_encoder_geography = OneHotEncoder()

geo_encoder = one_hot_encoder_geography.fit_transform(data[['Geography']])
geo_encoder

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 10000 stored elements and shape (10000, 3)>

In [6]:
geo_encoder.toarray()

array([[1., 0., 0.],
       [0., 0., 1.],
       [1., 0., 0.],
       ...,
       [1., 0., 0.],
       [0., 1., 0.],
       [1., 0., 0.]])

In [7]:
one_hot_encoder_geography.get_feature_names_out(['Geography'])

array(['Geography_France', 'Geography_Germany', 'Geography_Spain'],
      dtype=object)

In [8]:
geo_encoded_df = pd.DataFrame(geo_encoder.toarray(
), columns=one_hot_encoder_geography.get_feature_names_out(['Geography']))
geo_encoded_df

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0
1,0.0,0.0,1.0
2,1.0,0.0,0.0
3,1.0,0.0,0.0
4,0.0,0.0,1.0
...,...,...,...
9995,1.0,0.0,0.0
9996,1.0,0.0,0.0
9997,1.0,0.0,0.0
9998,0.0,1.0,0.0


In [9]:
#  Combine OneHot Encoded columns with the original data

data = pd.concat([data.drop('Geography', axis=1), geo_encoded_df], axis=1)
data.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


### 1. Pickle


In [10]:
# Save the encoders and scaler

with open('label_encoder_gender.pkl', 'wb') as file:
    pickle.dump(label_encoder_gender, file)

with open('one_hot_encoder_geography.pkl', 'wb') as file:
    pickle.dump(one_hot_encoder_geography, file)

### 2. .joblib


In [11]:
#  Save in .joblib
# import joblib

# joblib.dump(
#     label_encoder_gender,
#     "label_encoder_gender.joblib"
# )

# joblib.dump(
#     one_hot_encoder_geography,
#     "one_hot_encoder_geography.joblib"
# )

### 3. .skops


In [12]:
# pip install skops

In [13]:
# import skops.io as sio

# sio.dump(
#     label_encoder_gender,
#     "label_encoder_gender.skops"
# )

# sio.dump(
#     one_hot_encoder_geography,
#     "one_hot_encoder_geography.skops"
# )

In [14]:
data.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


## 4. Train-Test Split and Feature Scaling

Once the categorical features have been transformed, I will separate the input features (`X`) from the target (`y`).

I am using an 80/20 train-test split and applying `StandardScaler` so that the numerical features are on comparable scales before they are passed to the neural network.

In [15]:
#  Divide the dataset into independent and dependent features

X = data.drop('Exited', axis=1)
y = data['Exited']


#  Split the data in training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)


# Scale these features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [16]:
X_train

array([[ 0.35649971,  0.91324755, -0.6557859 , ...,  1.00150113,
        -0.57946723, -0.57638802],
       [-0.20389777,  0.91324755,  0.29493847, ..., -0.99850112,
         1.72572313, -0.57638802],
       [-0.96147213,  0.91324755, -1.41636539, ..., -0.99850112,
        -0.57946723,  1.73494238],
       ...,
       [ 0.86500853, -1.09499335, -0.08535128, ...,  1.00150113,
        -0.57946723, -0.57638802],
       [ 0.15932282,  0.91324755,  0.3900109 , ...,  1.00150113,
        -0.57946723, -0.57638802],
       [ 0.47065475,  0.91324755,  1.15059039, ..., -0.99850112,
         1.72572313, -0.57638802]])

In [17]:
with open('scaler.pkl', 'wb') as file:
    pickle.dump(scaler, file)

In [18]:
data

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,771,1,39,5,0.00,2,1,0,96270.64,0,1.0,0.0,0.0
9996,516,1,35,10,57369.61,1,1,1,101699.77,0,1.0,0.0,0.0
9997,709,0,36,7,0.00,1,0,1,42085.58,1,1.0,0.0,0.0
9998,772,1,42,3,75075.31,2,1,0,92888.52,1,0.0,1.0,0.0


## 5. Building the Artificial Neural Network

Now that the data is prepared, I can build the ANN.

I am keeping the architecture relatively simple because the purpose of this project is to understand the complete ANN workflow on tabular data rather than to create an unnecessarily complex network.

The network uses two hidden layers followed by a single sigmoid output neuron for binary classification.

In [19]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard
import datetime

2026-09-30 20:18:02.474986: I external/local_tsl/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-09-30 20:18:07.125208: I external/local_tsl/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-09-30 20:18:10.208056: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:479] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-09-30 20:18:15.385810: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:10575] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-09-30 20:18:15.386299: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1442] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-09-30 20:18:22.016353: I tensorflow/core/platform/cpu_feature_guard.cc:

In [20]:
(X_train.shape[1],)

(12,)

In [21]:
# Build Our ANN model

model = Sequential([
    # HL-1 connected with input layer
    Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
    Dense(32, activation='relu'),   # HL-2
    Dense(1, activation='sigmoid')  # Output Layer
])

/workspaces/Ann-churn-prediction/.venv/lib/python3.12/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [22]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │           832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,945 (11.50 KB)

 Trainable params: 2,945 (11.50 KB)

 Non-trainable params: 0 (0.00 B)

In [23]:
import tensorflow
opt = tensorflow.keras.optimizers.Adam(learning_rate=0.01)
loss = tensorflow.keras.losses.BinaryCrossentropy()
loss

<LossFunctionWrapper(<function binary_crossentropy at 0x78385ef81bc0>, kwargs={'from_logits': False, 'label_smoothing': 0.0, 'axis': -1})>

### Model Compilation

For binary churn prediction, I am using:

- Adam optimizer
- Binary Crossentropy loss
- Accuracy as the training metric

These choices give me a straightforward setup for learning how the network behaves during training.

In [24]:
# Compile the model

model.compile(optimizer=opt, loss="binary_crossentropy", metrics=['accuracy'])

## 6. Experiment Tracking with TensorBoard

Training a neural network is not only about the final score. I also want to see how the model behaves while it is learning.

For that reason, I am logging the training process with TensorBoard so that I can inspect metrics such as training and validation loss and accuracy over time.

In [25]:
# Setup the Tensorboard
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard

log_dir = "logs/fit/" + datetime.datetime.now().strftime("%y%m%d-%H%M%S")
tensorflow_callback = TensorBoard(log_dir=log_dir, histogram_freq=1)

### Early Stopping

I am also using Early Stopping to prevent the model from continuing to train once validation performance stops improving.

The best validation weights are restored at the end, so the final model keeps the strongest version observed during training.

In [26]:
# Setup EarlyStopping

early_Stopping_callback = EarlyStopping(
    monitor='val_loss', patience=10, restore_best_weights=True)

## 7. Training the Model

At this point, the data pipeline and model are ready.

I will now train the ANN for up to 100 epochs while monitoring validation loss. Early Stopping will decide when the model has stopped improving sufficiently.

In [27]:
# Training the model

history = model.fit(
    X_train, y_train, validation_data=(X_test, y_test), epochs=100,
    callbacks=[tensorflow_callback, early_Stopping_callback]
)

Epoch 1/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.8386 - loss: 0.3957 - val_accuracy: 0.8600 - val_loss: 0.3437
Epoch 2/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8546 - loss: 0.3557 - val_accuracy: 0.8520 - val_loss: 0.3619
Epoch 3/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8581 - loss: 0.3470 - val_accuracy: 0.8640 - val_loss: 0.3395
Epoch 4/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8606 - loss: 0.3428 - val_accuracy: 0.8550 - val_loss: 0.3505
Epoch 5/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8586 - loss: 0.3420 - val_accuracy: 0.8605 - val_loss: 0.3408
Epoch 6/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8620 - loss: 0.3380 - val_accuracy: 0.8625 - val_loss: 0.3407
Epoch 7/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8614 - loss: 0.3351 - val_accuracy: 0.8610 - val_loss: 0.3489
Epoch 8/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8635 - loss: 0.3344 - val_accu

## 8. Saving the Trained Model

The trained model needs to be available outside this notebook because the next step is inference.

I am therefore saving the ANN to disk along with the preprocessing objects used during training. Keeping these preprocessing objects is important because new customer data must go through the same transformations before being passed to the model.

In [28]:
model.save('model.h5')

In [29]:
## Load TensorBoard Extension 

%load_ext tensorboard

In [30]:
%reload_ext tensorboard

In [31]:
%tensorboard --logdir logs/fit

In [32]:
# !kill 7083

In [33]:
# %tensorboard --logdir logs

## 9. Next Step: Inference

The training stage is now complete.

In the next notebook, I will load the saved ANN and preprocessing objects and use them to generate a prediction for a new customer profile.

The important part is that the new input must follow the same preprocessing and feature ordering used during training.